## Install dependencies
and download sample dataset

In [ ]:
# Install library
!pip install -U transformers
# !pip install accelerate
# Download Dataset
!wget -P /kaggle/working https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip
!wget -P /kaggle/working https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip
!unzip /kaggle/working/Flickr8k_Dataset.zip
!unzip /kaggle/working/Flickr8k_text.zip

!echo "Downloaded Flickr8k dataset successfully."
!mkdir -p /kaggle/working/saved_model
!mkdir -p /kaggle/working/uploaded

## Imports

In [1]:
import os
from transformers import AutoProcessor, CLIPVisionModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import numpy as np
from huggingface_hub import login
import urllib.request

2025-04-14 23:16:15.109377: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744672575.312874      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744672575.372717      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
HF_HOME="/kaggle/working/huggingface" #@param {type:"string"}
HUGGINGFACE_TOKEN = "<insert your HF token here>" #@param {"type":"string"}
login(HUGGINGFACE_TOKEN)

## Vision embedding -> text embedding
projection layer

In [ ]:
class MyAdaptor(nn.Module) :
  def __init__(self, vis_token_embedding_size, word_embedding_size) :
    super(MyAdaptor, self).__init__()
    self.vis_token_embedding_size = vis_token_embedding_size
    self.word_embedding_size = word_embedding_size
    self.hidden_size = word_embedding_size//2

    self.fc1=nn.Linear(self.vis_token_embedding_size, self.hidden_size)
    #self.relu=nn.ReLU()
    #self.silu=nn.SiLU()
    self.gelu=nn.GELU()
    #self.dropout=nn.Dropout(0.1)
    self.adapter_linear = nn.Linear(self.hidden_size, self.word_embedding_size)

  def forward(self, img_output) :
    self.fc1.to(img_output.device)
    self.gelu.to(img_output.device)
    #self.dropout.to(img_output.device)
    self.adapter_linear.to(img_output.device)
    img_output = self.fc1(img_output)
    img_output = self.gelu(img_output)
    #img_output = self.dropout(img_output)
    #self.adapter_linear.to(img_output.device)
    img_embed = self.adapter_linear(img_output)
    return img_embed

## Define the model
Pass the image embedding as a fake "token" into the LLM prompt

In [4]:
class MyModel(nn.Module) :
  def __init__(self) :
    super(MyModel, self).__init__()
    self.model_language = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it", torch_dtype=torch.bfloat16, device_map="auto", attn_implementation='eager')
    self.tokenizer_language = AutoTokenizer.from_pretrained("google/gemma-2-2b-it", padding_side= 'right')
    self.image_processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32").image_processor
    self.model_image = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32", device_map="auto")

    self.word_embedding_size = 2304 # 2304 for 2b, 3584 for 9b, 4608 for 27b
    self.num_vocab = 256000

    self.trigger_str_img = "<start_image>"
    self.num_vis_token_summary = 50
    self.vis_token_embedding_size = 768
    self.adaptor = MyAdaptor(self.vis_token_embedding_size,self.word_embedding_size )
    self.dummy_img_token = (" ".join(["the"]*self.num_vis_token_summary)).strip()

  def search_trigger_idx(self, text_token, trigger_str) :
    all_token = text_token
    all_string_now = ""
    all_token_now = []
    dummy_start_token = None
    for token_idx in range(len(all_token)) :
      token_now = int(all_token[token_idx].detach().cpu().numpy())
      all_token_now.append(token_now)
      token_as_string = self.tokenizer_language.batch_decode([all_token_now],skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

      if trigger_str in token_as_string :
        dummy_start_token = token_idx + 1
        break
    return dummy_start_token

  def get_image_embed(self, image_input) :
    img_output = self.model_image(image_input)['last_hidden_state']
    img_embed = self.adaptor(img_output)

    return img_embed


  def replace_embedding_hook(self, image_input) :
    image_feature = self.get_image_embed(image_input)
    assert len(image_feature) == 1

    def now_hook(model, input, output) :
      real_input = input[0]
      batch_size, token_len = real_input.shape
      if(token_len > 1) :
        assert batch_size == 1
        dummy_start_token = self.search_trigger_idx(real_input[0], self.trigger_str_img )

        temp = image_feature[0]
        output[:,dummy_start_token:dummy_start_token+self.num_vis_token_summary] = temp
      return output
    return now_hook



  def split_and_replace(self, now_input_tokens, replacement_embed, start_loc) :
    num_token = len(replacement_embed)

    start_embed = now_input_tokens[0:start_loc]
    end_embed = now_input_tokens[start_loc+num_token:]
    replaced_embed = torch.cat((start_embed, replacement_embed.to(now_input_tokens.dtype), end_embed),0)

    return replaced_embed

  def forward_loss(self, image_input_raw, caption_output_raw) :
    instruction_now =  "<start_of_turn>user\n"
    instruction_now += f"<start_image> {self.dummy_img_token}\n<end_image>\n"
    instruction_now += f"Create a simple description of the image!\n<end_of_turn>\n<start_of_turn>model\n"

    image_input = self.image_processor(image_input_raw, return_tensors="pt")['pixel_values']
    image_input = image_input.to(device)

    caption_output = self.tokenizer_language(caption_output_raw,padding=True,return_tensors="pt")
    caption_output['input_ids'] = caption_output['input_ids'].to(device)
    caption_output['attention_mask'] = caption_output['attention_mask'].to(device)

    img_output = self.model_image(image_input)['last_hidden_state']
    img_embed = self.adaptor(img_output)

    all_text_with_prompt = [instruction_now + temp_text for temp_text in self.tokenizer_language.batch_decode(caption_output['input_ids'], skip_special_tokens=True)]
    all_tokens_with_prompt = self.tokenizer_language(all_text_with_prompt, padding=True, return_tensors="pt")
    all_tokens_with_prompt['input_ids'] = all_tokens_with_prompt['input_ids'].to(device).detach()
    all_tokens_with_prompt['attention_mask'] = all_tokens_with_prompt['attention_mask'].to(device).detach()

    all_token_prompt_embed = self.model_language.model.embed_tokens(all_tokens_with_prompt['input_ids'])
    prompt_len = len(self.tokenizer_language([instruction_now])['input_ids'][0])
    caption_label_now = all_tokens_with_prompt['input_ids'][:,prompt_len:]
    caption_label_now = F.one_hot(caption_label_now,self.num_vocab)
    attn_mask_now = all_tokens_with_prompt['attention_mask'][:,prompt_len:]

    ## Now replace placeholder tokens with image features and embed the lot

    all_replaced_feature = []
    for temp_idx in range(len(all_tokens_with_prompt['input_ids'])) :
      tokens_text_now = all_tokens_with_prompt['input_ids'][temp_idx].detach().cpu()
      dummy_location_caption = self.search_trigger_idx(tokens_text_now, self.trigger_str_img )
      image_replaced_prompt = self.split_and_replace(all_token_prompt_embed[temp_idx], img_embed[temp_idx], dummy_location_caption)

      all_replaced_feature.append(image_replaced_prompt)
    all_replaced_feature = torch.stack(all_replaced_feature)

    ## Proceed to compute LLM loss, apply softmax etc


    logits_now = self.model_language(inputs_embeds =all_replaced_feature, attention_mask=all_tokens_with_prompt['attention_mask'])

    logits_now = logits_now['logits']
    caption_prediction_now = logits_now[:,prompt_len-1:-1]
    caption_prediction_now = torch.softmax(caption_prediction_now,-1)
    caption_prediction_now = torch.maximum(caption_prediction_now,torch.as_tensor(1e-10).to(caption_prediction_now.dtype))
    caption_prediction_now = torch.minimum(caption_prediction_now,torch.as_tensor(1 - 1e-10).to(caption_prediction_now.dtype))


    loss_lm = -torch.sum(caption_label_now*torch.log(caption_prediction_now),-1)
    loss_lm = torch.sum(loss_lm*attn_mask_now,-1)/torch.sum(attn_mask_now,-1)
    loss_lm = torch.mean(loss_lm)

    return loss_lm

  def generate_aswer_image(self, input_string, pil_image, max_new_tokens = 32, do_sample=True, top_k=50, top_p=0.95, temperature =1 ) :

    input_with_dummy_prompt = self.tokenizer_language.apply_chat_template(input_string, tokenize=False, add_generation_prompt=True)
    input_with_dummy_prompt = input_with_dummy_prompt.replace("<image>", "<start_image> "+self.dummy_img_token+"\n<end_image>")
    dummy_input = self.tokenizer_language(input_with_dummy_prompt,padding=True,return_tensors="pt")
    dummy_input['input_ids'] = dummy_input['input_ids'].to(device)
    dummy_input['attention_mask'] = dummy_input['attention_mask'].to(device)
    assert len(dummy_input['input_ids']) == 1

    handler_image = None

    contains_image = False
    if self.trigger_str_img in input_with_dummy_prompt :
      image_input = self.image_processor([pil_image], return_tensors="pt")['pixel_values'].to(device)
      hook_now_image = self.replace_embedding_hook(image_input)
      contains_image = True
      handler_image = self.model_language.model.embed_tokens.register_forward_hook(hook_now_image)



    output_now = self.model_language.generate(**dummy_input,
                                              max_new_tokens = max_new_tokens,
                                              do_sample=do_sample,
                                              temperature=temperature,
                                              top_k=top_k,
                                              top_p=top_p,
                                              )
    output_string = self.tokenizer_language.batch_decode(output_now, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    if contains_image :
      handler_image.remove()

    return output_string.split("model\n")[-1]

## Train the model

In [5]:
CAPTION_PATH = "/kaggle/working/Flickr8k.token.txt"
IMAGES_FILE_PATH = "/kaggle/working/Flicker8k_Dataset"
SAVED_PATH = "/kaggle/working/saved_model/adaptor_caption.pt"

BATCH_SIZE = 8
NUM_ITERATION = 2000
SAVE_EVERY = 200
LEARNING_RATE = 1e-4
TRAIN_DATA_NUM = 7500

device = 'cpu'
if torch.cuda.is_available() :
  device = 'cuda'

In [6]:
def check_model_nan(model) :
  num_nan = 0
  for param in model.parameters() :
    num_nan += torch.sum(torch.isnan(param))
  return num_nan > 0

def getLabelDictionary(file_path) :
  file_now = open(file_path)
  all_string = file_now.read()
  all_string = all_string.split('\n')
  label_dictionary = {}
  for line_now in all_string :
    splitted_line = line_now.split('\t')
    if len(splitted_line) > 1 :
      file_name_now = splitted_line[0].split('#')[0]
      number_now = splitted_line[0].split('#')[1]
      label_now = splitted_line[1]

      if file_name_now in label_dictionary.keys() :
        label_dictionary[file_name_now].append(label_now)
      else :
        label_dictionary.update({
            file_name_now:[label_now]
        })

  return label_dictionary

def count_model_param(model_now) :
  counter = 0
  for param in model_now.parameters() :
    counter = counter + torch.sum(torch.ones_like(param))
  return counter

def sample_data_caption(file_list_now, caption_dict_now, n) :
  # base_path = "/content/Flicker8k_Dataset"
  base_path = IMAGES_FILE_PATH
  rand_idx = np.random.randint(0,len(file_list_now), n)

  all_image = []
  all_text = []
  for idx_now in rand_idx :
    file_now = base_path + "/" + file_list_now[idx_now]
    image_now = Image.open(file_now)
    all_image.append(image_now)

    text_list_now = caption_dict_now[file_list_now[idx_now]]
    selected_text_now_idx = np.random.randint(0,len(text_list_now))
    all_text.append(text_list_now[selected_text_now_idx])

  return all_image, all_text

In [7]:
label_dictionary = getLabelDictionary(CAPTION_PATH)
all_file = os.listdir(IMAGES_FILE_PATH)

In [8]:
model = MyModel()
#model = model.to(device)
model = model.to(torch.bfloat16)

for param in model.parameters() :
  param.requires_grad = True
for param in model.model_language.parameters() :
  param.requires_grad = False
for param in model.model_image.parameters() :
  param.requires_grad = False

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [9]:
optim = torch.optim.AdamW(model.parameters(),LEARNING_RATE)

In [ ]:
model.train()
best_loss = 1e6
best_checkpoint=0
for itr in range(NUM_ITERATION) :
  rand_image, rand_targets = sample_data_caption(all_file[0:TRAIN_DATA_NUM], label_dictionary, BATCH_SIZE)
  loss = model.forward_loss(rand_image, rand_targets)
  optim.zero_grad()
  loss.backward()
  optim.step()

  print(f"{itr+1}/{NUM_ITERATION}: {loss}", end="\r")

  if itr % SAVE_EVERY == (SAVE_EVERY - 1) :
    print("\nMODEL SAVED!")
    torch.save(model.adaptor.state_dict(),SAVED_PATH)
    if loss<best_loss :
      best_loss = loss
      best_checkpoint = itr+1
    print(f"BEST LOSS:{best_loss} at BEST CHECKPOINT:{best_checkpoint}")

## Infer with VLM

In [ ]:
image_url = "https://i.ytimg.com/vi/FHytoCvj90w/maxresdefault.jpg"
saved_name = "./temp_img." + image_url.split(".")[-1]
urllib.request.urlretrieve(image_url,saved_name)
pil_image_now = Image.open(saved_name)
pil_image_now

In [ ]:
output_string = model.generate_aswer_image([{
    "role":"user",
    "content":"<image>Describe this image accurately"
}], pil_image_now, max_new_tokens=256, top_k=40, top_p=0.95, temperature=0.4)
print(output_string)